# 1 · Explorar los datos de AeroLab

Antes de entrenar nada hay que **entender los datos**: qué columnas hay, cuántas filas de cada situación y
cómo se ve cada escenario. El conjunto `aerolab` lo ha escrito Node-RED (flujo *Serie 4 · AeroLab · Recoger datos*).

La columna **`escenario`** es la *etiqueta*: lo que estaba pasando de verdad (la fijan los botones del flujo).
En una planta real casi nunca se tiene; aquí la usamos para **comprobar** si los modelos aciertan.

In [ ]:
from labia import datos
import matplotlib.pyplot as plt

df = datos("aerolab")
print(len(df), "filas desde", df.ts.min(), "hasta", df.ts.max())
df.head()

In [ ]:
df["escenario"].value_counts()

In [ ]:
df.describe().round(2)

## La curva de potencia

La relación entre **viento** y **potencia** es la firma de un aerogenerador: crece con el cubo del viento hasta
la potencia nominal (5000 kW) y ahí se queda. Cada color es un escenario.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for esc, g in df.groupby("escenario"):
    ax.scatter(g.viento, g.potencia, s=6, alpha=0.5, label=esc)
ax.set_xlabel("viento (m/s)"); ax.set_ylabel("potencia (kW)"); ax.legend(); ax.grid(alpha=.3)
ax.set_title("Curva de potencia por escenario");

## La evolución en el tiempo

In [ ]:
fig, ejes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
for ax, col in zip(ejes, ["viento", "potencia", "vibracion", "temp_rodamiento"]):
    ax.plot(df.ts, df[col], lw=.8); ax.set_ylabel(col); ax.grid(alpha=.3)
# sombrear lo que no es normal
for ax in ejes:
    for _, tramo in df[df.escenario != "normal"].groupby((df.escenario != df.escenario.shift()).cumsum()):
        ax.axvspan(tramo.ts.iloc[0], tramo.ts.iloc[-1], color="orange", alpha=.15)
ejes[0].set_title("En naranja, los escenarios que no son «normal»");

## Correlaciones (solo en funcionamiento normal)

In [ ]:
normal = df[df.escenario == "normal"]
normal.drop(columns=["ts", "escenario"]).corr().round(2)

## ✏️ Preguntas

1. ¿Qué escenarios se distinguen a simple vista en la **curva de potencia**? ¿Y en la **vibración**?
2. La góndola desalineada **no activa ninguna alarma** de la máquina (mira la columna `averia`). ¿Cómo la verías tú?
3. ¿Qué variables están muy correlacionadas con la potencia? ¿Tiene sentido físico?
4. ¿Hay suficientes filas de cada escenario para entrenar y para evaluar?